# Amplificação Causal — Crase e Clíticos

Testa se **amplificar** features morfossintáticas produz efeitos causais mensuráveis:
- Amplificar Feature 4584 (crase) → modelo insere crases indevidas?
- Amplificar Feature 2817 (ênclise) → modelo prefere ênclise?
- Amplificar Feature 6215 (próclise) → modelo prefere próclise?
- Ablação combinada de TODAS features de crase (4584 + 2294) → crase desaparece?

**Tempo:** ~20 min no T4 | **Output:** `exp_amplificacao_results.json`

In [ ]:
!pip install transformer_lens sae_lens -q

In [ ]:
import torch
import numpy as np
import json
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

from sae_lens import SAE, HookedSAETransformer

LAYER = 13
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"

print("Carregando Gemma 2 2B...")
model = HookedSAETransformer.from_pretrained_no_processing(
    "gemma-2-2b", device=device, dtype=torch.float16,
)
print(f"Modelo: {model.cfg.model_name}")

print(f"Carregando SAE: {SAE_ID}...")
sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=device)
HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"
print(f"SAE: {sae.cfg.d_sae} features")

tokenizer = model.tokenizer
SAVE_DIR = "./data/checkpoints/"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
def generate_with_steering(model, sae, tokenizer, prompt, feature_ids,
                          multipliers, max_new_tokens=100, temperature=0.7,
                          hook_name=HOOK_NAME):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    feature_ids_t = torch.tensor(feature_ids, device=device)
    multipliers_t = torch.tensor(multipliers, dtype=torch.float16, device=device)

    def steering_hook(value, hook):
        with torch.no_grad():
            sae_input = value
            sae_acts = sae.encode(sae_input)
            modified_acts = sae_acts.clone()
            for fid, mult in zip(feature_ids_t, multipliers_t):
                modified_acts[:, :, fid] = modified_acts[:, :, fid] * mult
            reconstructed = sae.decode(modified_acts)
            error = sae_input - sae.decode(sae_acts)
            return reconstructed + error

    generated = input_ids.clone()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model.run_with_hooks(
                generated, fwd_hooks=[(hook_name, steering_hook)],
            )
            next_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            if next_token.item() == tokenizer.eos_token_id:
                break
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)

    return tokenizer.decode(generated[0], skip_special_tokens=True)


def generate_baseline(model, tokenizer, prompt, max_new_tokens=100,
                      temperature=0.7):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(generated)
            next_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            if next_token.item() == tokenizer.eos_token_id:
                break
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)
    return tokenizer.decode(generated[0], skip_special_tokens=True)

## Experimento 1: Amplificação de Crase (Feature 4584)

Prompts onde crase NÃO deve aparecer. Se amplificar 4584 faz o modelo inserir crases indevidas → evidência causal.

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

ALL_RESULTS = {}

# Prompts onde crase NÃO é esperada (contextos masculinos, verbos, etc.)
CRASE_PROMPTS_NO_CRASE = [
    "Eu vou a casa dele para",          # "a casa dele" — sem crase (casa indeterminada)
    "Ele se referiu a esse problema",    # "a esse" — sem crase antes de demonstrativo masc
    "Ela começou a correr quando",       # "a correr" — sem crase antes de verbo
    "Nós fomos a um restaurante",        # "a um" — sem crase antes de artigo indefinido
    "Ele disse a ela que",               # "a ela" — sem crase antes de pronome pessoal
    "A partir de amanhã, o governo",     # "A partir de" — sem crase
]

# Prompts onde crase É esperada
CRASE_PROMPTS_WITH_CRASE = [
    "A menina foi à escola e",           # "à escola" — crase correta
    "Ele se dedicou à pesquisa",         # "à pesquisa" — crase correta
    "Graças à chuva, a plantação",       # "à chuva" — crase correta
    "A empresa atendeu à demanda",       # "à demanda" — crase correta
]

MULTIPLIERS = [1.0, 4.0, 8.0, 12.0]

crase_results = {"amplification": [], "ablation_combined": []}

print("=== AMPLIFICAÇÃO DE CRASE (Feature 4584) ===\n")

for mult in MULTIPLIERS:
    print(f"\n--- Multiplier: {mult}× ---")
    mult_data = {"multiplier": mult, "no_crase_prompts": [], "with_crase_prompts": []}

    for prompt in CRASE_PROMPTS_NO_CRASE:
        torch.manual_seed(42)
        torch.cuda.manual_seed_all(42)

        if mult == 1.0:
            text = generate_baseline(model, tokenizer, prompt,
                                     max_new_tokens=60, temperature=0.3)
        else:
            text = generate_with_steering(model, sae, tokenizer, prompt,
                                          feature_ids=[4584],
                                          multipliers=[mult],
                                          max_new_tokens=60, temperature=0.3)

        has_crase = "à" in text[len(prompt):] or "À" in text[len(prompt):]
        mult_data["no_crase_prompts"].append({
            "prompt": prompt,
            "generated": text,
            "has_crase_in_output": has_crase,
        })
        marker = " ← CRASE INSERIDA!" if has_crase else ""
        print(f"  [no-crase] {prompt[:40]}...")
        print(f"    → {text[len(prompt):len(prompt)+70]}...{marker}")

    for prompt in CRASE_PROMPTS_WITH_CRASE:
        torch.manual_seed(42)
        torch.cuda.manual_seed_all(42)

        if mult == 1.0:
            text = generate_baseline(model, tokenizer, prompt,
                                     max_new_tokens=60, temperature=0.3)
        else:
            text = generate_with_steering(model, sae, tokenizer, prompt,
                                          feature_ids=[4584],
                                          multipliers=[mult],
                                          max_new_tokens=60, temperature=0.3)

        has_crase = "à" in text[len(prompt):] or "À" in text[len(prompt):]
        mult_data["with_crase_prompts"].append({
            "prompt": prompt,
            "generated": text,
            "has_crase_in_output": has_crase,
        })
        print(f"  [with-crase] {prompt[:40]}...")
        print(f"    → {text[len(prompt):len(prompt)+70]}...")

    crase_results["amplification"].append(mult_data)

# Count hypercorrections per multiplier
print("\n\n=== RESUMO CRASE ===")
print(f"{'Mult':>6} | {'Crases indevidas':>18} | {'de':>4} prompts sem crase")
print("-" * 45)
for m in crase_results["amplification"]:
    n_hyper = sum(1 for p in m["no_crase_prompts"] if p["has_crase_in_output"])
    print(f"{m['multiplier']:>6.1f} | {n_hyper:>18} | {len(m['no_crase_prompts']):>4}")

## Experimento 2: Amplificação de Clíticos — Ênclise (2817) e Próclise (6215)

Usar prompts onde a norma pede próclise e amplificar a feature de ênclise (e vice-versa). Se a posição do pronome muda → causal.

In [ ]:
# Prompts que induzem PRÓCLISE (negação, subordinação, advérbio)
PROCLISE_PROMPTS = [
    "Não me disseram o motivo",           # negação → próclise
    "O professor que me ajudou",          # subordinação → próclise
    "Ninguém se lembrou de trazer",       # neg indefinido → próclise
    "Já lhe contei essa história",        # advérbio → próclise
    "Alguém me chamou quando eu",         # pronome indefinido → próclise
    "Nunca se soube a verdade sobre",     # negação → próclise
]

# Prompts que induzem ÊNCLISE (início de frase, imperativo)
ENCLISE_PROMPTS = [
    "Diga-me o que aconteceu",            # imperativo → ênclise
    "Faça-me um favor e",                 # imperativo → ênclise
    "Conte-nos a história do",            # imperativo → ênclise
    "Apresentou-se como o novo",          # início de frase → ênclise
    "Levantou-se cedo e foi",             # início de frase → ênclise
    "Deram-lhe o prêmio de",              # início de frase → ênclise
]

# Pronomes clíticos a detectar
CLITICS = ["me ", "te ", "se ", "lhe ", "nos ", "lhes ", "lo ", "la ", "los ", "las "]
ENCLISE_PATTERN = lambda t: sum(1 for c in CLITICS if f"-{c.strip()}" in t.lower())
PROCLISE_PATTERN = lambda t: sum(1 for c in CLITICS if c in t.lower() and f"-{c.strip()}" not in t.lower())

CLITIC_MULTS = [1.0, 4.0, 8.0, 12.0]

clitic_results = {"enclise_amplification": [], "proclise_amplification": []}

# Test 1: Amplificar ênclise (2817) em contextos de próclise
print("=== AMPLIFICAR ÊNCLISE (2817) EM CONTEXTOS DE PRÓCLISE ===\n")

for mult in CLITIC_MULTS:
    print(f"\n--- Multiplier: {mult}× ---")
    mult_data = {"multiplier": mult, "generations": []}

    for prompt in PROCLISE_PROMPTS:
        torch.manual_seed(42)
        torch.cuda.manual_seed_all(42)

        if mult == 1.0:
            text = generate_baseline(model, tokenizer, prompt,
                                     max_new_tokens=60, temperature=0.3)
        else:
            text = generate_with_steering(model, sae, tokenizer, prompt,
                                          feature_ids=[2817],
                                          multipliers=[mult],
                                          max_new_tokens=60, temperature=0.3)
        output = text[len(prompt):]
        n_enc = ENCLISE_PATTERN(output)
        n_proc = PROCLISE_PATTERN(output)
        mult_data["generations"].append({
            "prompt": prompt, "generated": text,
            "enclise_count": n_enc, "proclise_count": n_proc,
        })
        print(f"  {prompt[:45]}... → enc={n_enc} pro={n_proc}")
        print(f"    {output[:80]}...")

    clitic_results["enclise_amplification"].append(mult_data)

# Test 2: Amplificar próclise (6215) em contextos de ênclise
print("\n\n=== AMPLIFICAR PRÓCLISE (6215) EM CONTEXTOS DE ÊNCLISE ===\n")

for mult in CLITIC_MULTS:
    print(f"\n--- Multiplier: {mult}× ---")
    mult_data = {"multiplier": mult, "generations": []}

    for prompt in ENCLISE_PROMPTS:
        torch.manual_seed(42)
        torch.cuda.manual_seed_all(42)

        if mult == 1.0:
            text = generate_baseline(model, tokenizer, prompt,
                                     max_new_tokens=60, temperature=0.3)
        else:
            text = generate_with_steering(model, sae, tokenizer, prompt,
                                          feature_ids=[6215],
                                          multipliers=[mult],
                                          max_new_tokens=60, temperature=0.3)
        output = text[len(prompt):]
        n_enc = ENCLISE_PATTERN(output)
        n_proc = PROCLISE_PATTERN(output)
        mult_data["generations"].append({
            "prompt": prompt, "generated": text,
            "enclise_count": n_enc, "proclise_count": n_proc,
        })
        print(f"  {prompt[:45]}... → enc={n_enc} pro={n_proc}")
        print(f"    {output[:80]}...")

    clitic_results["proclise_amplification"].append(mult_data)

# Resumo
print("\n\n=== RESUMO CLÍTICOS ===")
print("\nAmplificar ênclise (2817) em contextos de próclise:")
print(f"{'Mult':>6} | {'Total ênclises':>15} | {'Total próclises':>16}")
print("-" * 45)
for m in clitic_results["enclise_amplification"]:
    te = sum(g["enclise_count"] for g in m["generations"])
    tp = sum(g["proclise_count"] for g in m["generations"])
    print(f"{m['multiplier']:>6.1f} | {te:>15} | {tp:>16}")

print("\nAmplificar próclise (6215) em contextos de ênclise:")
print(f"{'Mult':>6} | {'Total ênclises':>15} | {'Total próclises':>16}")
print("-" * 45)
for m in clitic_results["proclise_amplification"]:
    te = sum(g["enclise_count"] for g in m["generations"])
    tp = sum(g["proclise_count"] for g in m["generations"])
    print(f"{m['multiplier']:>6.1f} | {te:>15} | {tp:>16}")

## Experimento 3: Ablação combinada de features de crase

Ablação simultânea das features 4584 (crase) + 2294 (contração "do") → crase desaparece em contextos obrigatórios?

In [ ]:
CRASE_OBLIGATORY_PROMPTS = [
    "A aluna foi à biblioteca buscar",
    "Ele dedicou sua vida à ciência e",
    "O governo atendeu à solicitação dos",
    "A empresa se adequou à nova legislação",
    "O relatório faz referência à pesquisa de",
    "O professor recorreu à diretoria para",
    "A decisão coube à comissão avaliadora",
    "Ele entregou o documento à secretária",
]

ablation_combined_results = {"single_ablation": [], "combined_ablation": [], "baseline": []}

print("=== ABLAÇÃO COMBINADA: CRASE ===\n")

# Baseline
print("--- Baseline (sem intervenção) ---")
for prompt in CRASE_OBLIGATORY_PROMPTS:
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    text = generate_baseline(model, tokenizer, prompt, max_new_tokens=60, temperature=0.3)
    output = text[len(prompt):]
    has_crase = "à" in output or "À" in output
    ablation_combined_results["baseline"].append({
        "prompt": prompt, "generated": text, "has_crase": has_crase,
    })
    print(f"  {prompt[:50]}... → crase={'SIM' if has_crase else 'NÃO'}")
    print(f"    {output[:80]}...")

# Ablação só 4584
print("\n--- Ablação Feature 4584 (crase) apenas ---")
for prompt in CRASE_OBLIGATORY_PROMPTS:
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    text = generate_with_steering(model, sae, tokenizer, prompt,
                                  feature_ids=[4584],
                                  multipliers=[0.0],
                                  max_new_tokens=60, temperature=0.3)
    output = text[len(prompt):]
    has_crase = "à" in output or "À" in output
    ablation_combined_results["single_ablation"].append({
        "prompt": prompt, "generated": text, "has_crase": has_crase,
    })
    print(f"  {prompt[:50]}... → crase={'SIM' if has_crase else 'NÃO'}")
    print(f"    {output[:80]}...")

# Ablação combinada: 4584 + 2294
print("\n--- Ablação Combinada 4584 + 2294 ---")
for prompt in CRASE_OBLIGATORY_PROMPTS:
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    text = generate_with_steering(model, sae, tokenizer, prompt,
                                  feature_ids=[4584, 2294],
                                  multipliers=[0.0, 0.0],
                                  max_new_tokens=60, temperature=0.3)
    output = text[len(prompt):]
    has_crase = "à" in output or "À" in output
    ablation_combined_results["combined_ablation"].append({
        "prompt": prompt, "generated": text, "has_crase": has_crase,
    })
    print(f"  {prompt[:50]}... → crase={'SIM' if has_crase else 'NÃO'}")
    print(f"    {output[:80]}...")

# Resumo
baseline_crase = sum(1 for p in ablation_combined_results["baseline"] if p["has_crase"])
single_crase = sum(1 for p in ablation_combined_results["single_ablation"] if p["has_crase"])
combined_crase = sum(1 for p in ablation_combined_results["combined_ablation"] if p["has_crase"])
n = len(CRASE_OBLIGATORY_PROMPTS)

print(f"\n\n=== RESUMO ABLAÇÃO CRASE (de {n} prompts com crase obrigatória) ===")
print(f"  Baseline:           {baseline_crase}/{n} com crase")
print(f"  Ablação 4584:       {single_crase}/{n} com crase")
print(f"  Ablação 4584+2294:  {combined_crase}/{n} com crase")

## Experimento 4: Amplificação dose-resposta (crase)

Verifica se há relação monotônica entre intensidade de amplificação e frequência de crases no output.

In [ ]:
DOSE_MULTS = [1.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 16.0]

DOSE_PROMPTS = [
    "O aluno se preparou para o exame final de",
    "A empresa lançou um novo produto no mercado",
    "O governo anunciou uma nova política de",
    "Os cientistas publicaram os resultados da",
]

dose_results = []

print("=== DOSE-RESPOSTA CRASE (Feature 4584) ===\n")

for mult in DOSE_MULTS:
    crase_counts = []
    total_tokens = 0

    for prompt in DOSE_PROMPTS:
        torch.manual_seed(42)
        torch.cuda.manual_seed_all(42)

        if mult == 1.0:
            text = generate_baseline(model, tokenizer, prompt,
                                     max_new_tokens=80, temperature=0.3)
        else:
            text = generate_with_steering(model, sae, tokenizer, prompt,
                                          feature_ids=[4584],
                                          multipliers=[mult],
                                          max_new_tokens=80, temperature=0.3)

        output = text[len(prompt):]
        n_crase = output.count("à") + output.count("À")
        n_tokens = len(tokenizer.encode(output))
        crase_counts.append(n_crase)
        total_tokens += n_tokens

    total_crases = sum(crase_counts)
    crases_per_100 = (total_crases / max(total_tokens, 1)) * 100

    dose_results.append({
        "multiplier": mult,
        "total_crases": total_crases,
        "total_tokens": total_tokens,
        "crases_per_100_tokens": round(crases_per_100, 2),
        "per_prompt": crase_counts,
    })
    print(f"  {mult:>5.1f}× → {total_crases:>3} crases em {total_tokens:>4} tokens ({crases_per_100:.2f}/100tok)")

# Verificar monotonicidade
vals = [d["crases_per_100_tokens"] for d in dose_results]
is_monotonic = all(vals[i] <= vals[i+1] for i in range(len(vals)-1))
print(f"\n  Relação monotônica: {'SIM' if is_monotonic else 'NÃO (mas tendência crescente?)'}")
print(f"  Tendência: {vals}")

## Experimento 5: Ablação combinada de clíticos (2817 + 6215 + 15135)

Se clitics estão distribuídos, ablação conjunta de TODAS as features de clíticos deve produzir efeito mais forte.

In [ ]:
CLITIC_RICH_PROMPTS = [
    "Eu não me lembro de quando",
    "Ela pediu-me que fizesse",
    "Ninguém se atreveu a",
    "Já lhe contaram sobre o",
    "O relatório apresentou-se como",
    "Disseram-lhe que o prazo",
    "Nós nos preparamos para a",
    "Quem me chamou foi o",
]

ALL_CLITIC_FEATURES = [2817, 6215, 15135]

clitic_ablation_results = {"baseline": [], "single_2817": [], "single_6215": [],
                           "single_15135": [], "combined_all": []}

print("=== ABLAÇÃO DE CLÍTICOS ===\n")

configs = [
    ("baseline", [], []),
    ("single_2817", [2817], [0.0]),
    ("single_6215", [6215], [0.0]),
    ("single_15135", [15135], [0.0]),
    ("combined_all", ALL_CLITIC_FEATURES, [0.0, 0.0, 0.0]),
]

for config_name, fids, mults in configs:
    print(f"\n--- {config_name} ---")
    total_enc = 0
    total_proc = 0

    for prompt in CLITIC_RICH_PROMPTS:
        torch.manual_seed(42)
        torch.cuda.manual_seed_all(42)

        if not fids:
            text = generate_baseline(model, tokenizer, prompt,
                                     max_new_tokens=60, temperature=0.3)
        else:
            text = generate_with_steering(model, sae, tokenizer, prompt,
                                          feature_ids=fids,
                                          multipliers=mults,
                                          max_new_tokens=60, temperature=0.3)
        output = text[len(prompt):]
        n_enc = ENCLISE_PATTERN(output)
        n_proc = PROCLISE_PATTERN(output)
        total_enc += n_enc
        total_proc += n_proc

        clitic_ablation_results[config_name].append({
            "prompt": prompt, "generated": text,
            "enclise_count": n_enc, "proclise_count": n_proc,
        })
        print(f"  {prompt[:45]}... → enc={n_enc} pro={n_proc}")

    print(f"  TOTAL: ênclises={total_enc} próclises={total_proc}")

# Resumo comparativo
print("\n\n=== RESUMO COMPARATIVO CLÍTICOS ===")
print(f"{'Config':<20} | {'Ênclises':>10} | {'Próclises':>10} | {'Total clíticos':>15}")
print("-" * 65)
for config_name, _, _ in configs:
    te = sum(g["enclise_count"] for g in clitic_ablation_results[config_name])
    tp = sum(g["proclise_count"] for g in clitic_ablation_results[config_name])
    print(f"{config_name:<20} | {te:>10} | {tp:>10} | {te+tp:>15}")

## Salvar resultados

In [ ]:
ALL_AMPLIFICATION_RESULTS = {
    "experiment": "Amplificação causal — crase e clíticos",
    "model": "gemma-2-2b",
    "layer": LAYER,
    "crase": {
        "amplification": crase_results["amplification"],
        "ablation_combined": ablation_combined_results,
        "dose_response": dose_results,
    },
    "clitics": {
        "amplification_enclise": clitic_results["enclise_amplification"],
        "amplification_proclise": clitic_results["proclise_amplification"],
        "ablation_combined": clitic_ablation_results,
    },
}

output_path = os.path.join(SAVE_DIR, "exp_amplificacao_results.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(ALL_AMPLIFICATION_RESULTS, f, ensure_ascii=False, indent=2)

print(f"Resultados salvos em {output_path}")
print(f"Tamanho: {os.path.getsize(output_path)/1024:.1f} KB")

print("\n=== ANÁLISE FINAL ===")
print("\nCRASE:")
dose_vals = [d["crases_per_100_tokens"] for d in dose_results]
print(f"  Dose-resposta (crases/100tok): {dose_vals}")
baseline_c = sum(1 for p in ablation_combined_results["baseline"] if p["has_crase"])
single_c = sum(1 for p in ablation_combined_results["single_ablation"] if p["has_crase"])
combined_c = sum(1 for p in ablation_combined_results["combined_ablation"] if p["has_crase"])
print(f"  Ablação: baseline={baseline_c}/{n} → single={single_c}/{n} → combined={combined_c}/{n}")

print("\nCLÍTICOS:")
for config_name, _, _ in configs:
    te = sum(g["enclise_count"] for g in clitic_ablation_results[config_name])
    tp = sum(g["proclise_count"] for g in clitic_ablation_results[config_name])
    print(f"  {config_name}: ênclise={te} próclise={tp} total={te+tp}")

print("\n✓ Notebook completo!")